In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
# 自动定位项目根目录
cwd = os.getcwd()
if os.path.basename(cwd) == 'notes':
    os.chdir('..')
print('当前工作目录:', os.getcwd())

In [ ]:
# ============================================================
# XGBoost 时序预测模型
# 严格按照官方文档 API: DMatrix → xgb.train → bst.predict
# 文档: https://xgboost.readthedocs.io/en/latest/python/python_intro.html
# ============================================================

# --- 1. 加载数据 ---
df_train = pd.read_csv('data/df_train.csv', index_col='datetime', parse_dates=True)
df_test  = pd.read_csv('data/df_test.csv',  index_col='datetime', parse_dates=True)

# 确保无缺失值 (XGBoost 本身支持缺失值, 但这里先清理保证干净)
df_train = df_train.dropna(subset=['pm_ave', 'TEMP', 'HUMI'])
df_test  = df_test.dropna(subset=['pm_ave', 'TEMP', 'HUMI'])

# --- 2. 构建自回归特征 ---
# 用过去 tau=24 步的 4 个通道 (pm_ave, TEMP, HUMI, hour) 预测下一步 pm_ave
# 注意: XGBoost 是树模型, 不需要标准化 (对单调变换不敏感)
tau = 24        # 回看窗口
n_features = 4  # pm_ave, TEMP, HUMI, hour

def build_features(df, tau=24, n_features=4):
    """构建滑窗特征矩阵: 每行 = 过去 tau 步 × n_features 通道, 展平为一维向量"""
    pm   = df['pm_ave'].values
    temp = df['TEMP'].values
    humi = df['HUMI'].values
    hour = df.index.hour.values
    n = len(df)
    X = np.zeros((n - tau, tau * n_features))
    for t in range(tau):
        X[:, t * n_features + 0] = pm[t: n - tau + t]     # pm_ave
        X[:, t * n_features + 1] = temp[t: n - tau + t]   # TEMP
        X[:, t * n_features + 2] = humi[t: n - tau + t]   # HUMI
        X[:, t * n_features + 3] = hour[t: n - tau + t]   # hour
    y = pm[tau:]  # label = 下一时刻的 pm_ave
    return X, y

X_train, y_train = build_features(df_train, tau)
X_test,  y_test  = build_features(df_test,  tau)
print(f'训练集: {X_train.shape}  |  验证集: {X_test.shape}')

# --- 3. 创建 DMatrix (XGBoost 专用数据结构, 官方推荐) ---
# DMatrix 针对内存效率和训练速度做了优化
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test,  label=y_test)

# --- 4. 设置参数 (官方文档写法) ---
params = {
    'max_depth': 6,                    # 树的最大深度, 控制模型复杂度
    'eta': 0.1,                        # 学习率 (步长收缩), 越小越保守
    'objective': 'reg:squarederror',   # 回归目标: 最小化均方误差
    'eval_metric': 'rmse',             # 评估指标: 均方根误差
    'subsample': 0.8,                  # 每棵树随机采样 80% 的行
    'colsample_bytree': 0.8,           # 每棵树随机采样 80% 的列
    'nthread': 4,                      # 并行线程数
}

# 指定验证集 (官方文档: evals 参数)
# 用 test 集作为验证集, 监控早停
evals = [(dtrain, 'train'), (dtest, 'eval')]

# --- 5. 训练 (带早停, 官方文档推荐) ---
# early_stopping_rounds: 验证误差连续 N 轮无改善则停止
# verbose_eval: 每 10 轮打印一次评估结果
num_boost_round = 200
bst = xgb.train(
    params, dtrain,
    num_boost_round=num_boost_round,
    evals=evals,
    early_stopping_rounds=20,
    verbose_eval=10,
)

# 官方文档: 早停后 bst.best_iteration 和 bst.best_score 可用
print(f'\n最佳迭代轮次: {bst.best_iteration}')
print(f'最佳验证 RMSE: {bst.best_score:.4f}')

# --- 6. 特征重要性可视化 (官方文档: xgb.plot_importance) ---
fig, ax = plt.subplots(figsize=(8, 6))
xgb.plot_importance(bst, max_num_features=20, ax=ax)
plt.title('Feature Importance (Top 20)')
plt.tight_layout()
plt.show()

# --- 7. 多步预测: 前 24 条公开, 剩余用预测值 ---
# TEMP/HUMI/hour 是外生变量 (测试集已知), 只有 pm_ave 需要多步预测
n_public = 24

# 提取测试集原始数据
pm_test   = df_test['pm_ave'].values
temp_test = df_test['TEMP'].values
humi_test = df_test['HUMI'].values
hour_test = df_test.index.hour.values

# 初始化预测数组: 前 24 条填入真实值 (公开)
test_preds = np.zeros(len(pm_test))
test_preds[:n_public] = pm_test[:n_public]

# 逐点预测: pm_ave 用预测值替代, 外生变量用真实值
for i in range(n_public, len(pm_test)):
    # 构建特征向量: 过去 tau 步 × 4 通道
    window = np.zeros(tau * n_features)
    for t in range(tau):
        idx = i - tau + t
        # pm_ave 通道: 公开区间用真实值, 之后用预测值
        if idx < n_public:
            window[t * n_features + 0] = pm_test[idx]
        else:
            window[t * n_features + 0] = test_preds[idx]
        # 外生变量: 始终用真实值
        window[t * n_features + 1] = temp_test[idx]
        window[t * n_features + 2] = humi_test[idx]
        window[t * n_features + 3] = hour_test[idx]
    # 用最佳模型预测 (官方文档: iteration_range 指定使用前 best_iteration+1 棵树)
    dpredict = xgb.DMatrix(window.reshape(1, -1))
    test_preds[i] = bst.predict(dpredict, iteration_range=(0, bst.best_iteration + 1))[0]

print(f'\n公开起点: 前 {n_public} 条  |  预测区间: 第 {n_public+1} ~ {len(pm_test)} 条 (共 {len(pm_test)-n_public} 条)')

In [ ]:
# --- 预测结果绘图 ---
# 前 24 条: predicted = actual (公开数据)
# 第 25 条起: 多步预测 (pm_ave 用预测值, TEMP/HUMI/hour 用真实值)
test_time = np.arange(1, len(pm_test) + 1)
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_time, pm_test, linewidth=0.8, label='actual')
ax.plot(test_time, test_preds, linewidth=0.8, label='predicted')
ax.axvline(x=n_public, color='r', linestyle='--', alpha=0.5, label=f'public end (t={n_public})')
ax.set_xlabel('Test timestep')
ax.set_ylabel('PM2.5 (μg/m³)')
ax.set_title('XGBoost — Multistep Prediction')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 预测结果表格 + MSE ---
# 只展示预测区间 (第 25 条起, 共 377 条)
df_result = pd.DataFrame({
    'timestamp': df_test.index[n_public:],
    'predicted': test_preds[n_public:],
    'actual':    pm_test[n_public:],
})
df_result['error'] = df_result['actual'] - df_result['predicted']

# 逐条计算均方误差
err = df_result['error']
mse = (err ** 2).mean()

print(f'预测样本数: {len(df_result)}')
print(f'MSE = {mse:.2f}')
df_result